In [ ]:
import random
import pandas as pd
from datetime import datetime, timedelta
from collections import deque, Counter

In [ ]:
#task1

In [ ]:

products = ["Laptop", "Phone", "Tablet", "Headphones", "Monitor"]
countries = ["usa", "germany", "france", "canada", "japan"]

raw_records = []

for i in range(200):
    record = {
        "order_id": i,
        "customer_id": random.randint(1, 30),
        "product_name": random.choice(products),
        "quantity": random.randint(1, 5),
        "unit_price": round(random.uniform(5, 500), 2),
        "order_date": str(datetime(2025,1,1) + timedelta(days=random.randint(0,180))),
        "shipping_country": random.choice(countries)
    }
    raw_records.append(record)

# introduce problems
for i in range(3):
    raw_records[i]["product_name"] = None

for i in range(3,6):
    raw_records[i]["quantity"] = -1

for i in range(6,9):
    raw_records[i]["order_date"] = "not-a-date"

for i in range(9,12):
    raw_records[i]["order_id"] = 1

for i in range(12,15):
    raw_records[i]["quantity"] = "five"

In [ ]:
raw_records

In [ ]:
def extract(raw_records):

    valid_records = []
    rejected_records = []

    reasons = {}

    for r in raw_records:

        reason = None

        if not r.get("product_name"):
            reason = "missing_product"

        elif not isinstance(r["quantity"], (int,float)) or not isinstance(r["unit_price"], (int,float)):
            reason = "invalid_numeric"

        elif r["quantity"] <= 0 or r["unit_price"] <= 0:
            reason = "negative_values"

        else:
            try:
                pd.to_datetime(r["order_date"])
            except:
                reason = "invalid_date"

        if reason:
            rejected_records.append({**r, "reason": reason})
            reasons[reason] = reasons.get(reason,0)+1
        else:
            valid_records.append(r)

    print("Valid:", len(valid_records))
    print("Rejected:", len(rejected_records))
    print("Reasons:", reasons)

    return valid_records, rejected_records

In [ ]:
extract(raw_records)

In [ ]:
valid_records, rejected_records = extract(raw_records)

In [ ]:
def transform(valid_records):

    df = pd.DataFrame(valid_records)

    df["order_date"] = pd.to_datetime(df["order_date"])

    df["total_amount"] = df["quantity"] * df["unit_price"]

    df["order_month"] = df["order_date"].dt.month

    df["order_day_of_week"] = df["order_date"].dt.day_name()

    df["shipping_country"] = df["shipping_country"].str.title()

    df = df.drop_duplicates(subset="order_id", keep="first")

    def categorize(x):
        if x < 25:
            return "small"
        elif x <= 100:
            return "medium"
        else:
            return "large"

    df["amount_category"] = df["total_amount"].apply(categorize)

    return df

In [ ]:
df_clean = transform(valid_records)
df_clean

In [ ]:
def load(df, path):

    df.to_parquet(path)

    df_loaded = pd.read_parquet(path)

    print("Saved rows:", len(df))
    print("Loaded rows:", len(df_loaded))

    print("Dtypes match:", df.dtypes.equals(df_loaded.dtypes))

    return df_loaded

In [ ]:
df_loaded = load(df_clean, "orders_clean.parquet")

In [ ]:
valid_records, rejected_records = extract(raw_records)

df_clean = transform(valid_records)

df_loaded = load(df_clean, "orders_clean.parquet")

print("Pipeline Summary")
print("Raw:", len(raw_records))
print("Valid:", len(valid_records))
print("Transformed:", len(df_clean))
print("Loaded:", len(df_loaded))

In [ ]:
#task2

In [ ]:
def extract_elt(raw):
    return raw

In [ ]:
df_raw = pd.DataFrame(extract_elt(raw_records))

df_raw = df_raw.astype(str)

df_raw.to_parquet("raw_lake.parquet")

In [ ]:
df_lake = pd.read_parquet("raw_lake.parquet")

records = df_lake.to_dict("records")

# convert numeric fields where possible
for r in records:
    try:
        r["quantity"] = float(r["quantity"])
    except:
        pass

    try:
        r["unit_price"] = float(r["unit_price"])
    except:
        pass

valid_records, rejected_records = extract(records)

df_elt = transform(valid_records)

In [ ]:
#ETL validates data before loading while ELT loads raw data first and applies validation during transformation.

In [ ]:
#In the ETL pipeline, invalid records were detected during the extraction stage before data was loaded into storage. This means only validated and cleaned data reached the destination dataset.

#In the ELT pipeline, most raw data was first loaded into the data lake and validation happened later during transformation. As a result, the storage layer contained raw and potentially invalid records.

#Advantages of ETL:
# Data is cleaned before storage
# Higher quality datasets
# Useful for structured warehouses

#Advantages of ELT:
# Raw data preserved for future reprocessing
# Faster ingestion
# Better for big data and data lakes

#ETL is preferred when strict data quality is required before loading. ELT is preferred in large-scale analytics systems where storage is cheap and transformations can run later.

In [ ]:
#task3

In [ ]:
database = {"orders": [], "features": {}}

def order_service_write(order):
    database["orders"].append(order)

def feature_service_compute():

    df = pd.DataFrame(database["orders"])

    features = df.groupby("customer_id").agg(
        total_orders=("order_id","count"),
        avg_amount=("total_amount","mean"),
        last_order=("order_date","max")
    )

    database["features"] = features.to_dict("index")

def prediction_service_read(customer_id):
    return database["features"].get(customer_id)

In [ ]:
def get_orders():
    return database["orders"]

def compute_features():

    df = pd.DataFrame(get_orders())

    return df.groupby("customer_id").agg(
        total_orders=("order_id","count"),
        avg_amount=("total_amount","mean")
    ).to_dict("index")

def prediction(customer_id):
    features = compute_features()
    return features.get(customer_id)

In [ ]:
class Broker:

    def __init__(self):
        self.topics = {}

    def publish(self, topic, message):
        self.topics.setdefault(topic, []).append(message)

    def subscribe(self, topic):
        return self.topics.get(topic, [])

broker = Broker()

In [ ]:
def order_service(order):
    broker.publish("orders", order)

In [ ]:
def feature_service():

    orders = broker.subscribe("orders")

    df = pd.DataFrame(orders)

    if len(df) == 0:
        return

    features = df.groupby("customer_id").size().to_dict()

    broker.publish("features", features)

In [ ]:
def prediction_service():
    return broker.subscribe("features")

In [ ]:
#While database-based and direct service-to-service communication models create tight coupling and increase the risk of immediate cascading failures, message brokers facilitate asynchronous interaction, which significantly enhances system resiliency and fault tolerance by allowing components to recover independently.

In [ ]:
#task4

In [ ]:
def batch_processing(df):

    daily = df.groupby(df["order_date"].dt.date).agg(
        total_orders=("order_id","count"),
        total_revenue=("total_amount","sum"),
        avg_order_size=("total_amount","mean"),
        unique_customers=("customer_id","nunique")
    )

    top_product = df.groupby(["order_date","product_name"])["total_amount"].sum().reset_index()

    return daily

In [ ]:
batch_results = batch_processing(df_clean)
batch_results.head()

In [ ]:


class StreamProcessor:

    def __init__(self):

        self.total_orders = 0
        self.total_revenue = 0

        self.window = deque(maxlen=50)

        self.customers = set()

        self.products = Counter()

    def process(self, order):

        amount = order["total_amount"]

        self.total_orders += 1
        self.total_revenue += amount

        self.window.append(order)

        self.customers.add(order["customer_id"])

        self.products[order["product_name"]] += 1

    def stats(self):

        avg_window = sum(o["total_amount"] for o in self.window) / len(self.window)

        top_product = self.products.most_common(1)

        return {
            "total_orders": self.total_orders,
            "total_revenue": self.total_revenue,
            "window_avg": avg_window,
            "unique_customers": len(self.customers),
            "top_product": top_product
        }

In [ ]:
stream = StreamProcessor()

for i, r in df_clean.iterrows():

    stream.process(r)

    if (i+1) % 50 == 0:
        print(stream.stats())

In [ ]:
#Batch processing calculates stable historical aggregates from entire datasets, while stream processing evaluates recent windows for real-time operational metrics, explaining why their resulting statistics naturally differ.

In [ ]:
#task5

In [ ]:
customers = df_clean["customer_id"].unique()[:5]

features = {}

for c in customers:

    cust_df = df_clean[df_clean["customer_id"]==c]

    batch_features = {
        "total_lifetime_orders": len(cust_df),
        "avg_order_amount": cust_df["total_amount"].mean(),
        "days_since_first_order": (pd.Timestamp.now() - cust_df["order_date"].min()).days,
        "most_purchased_category": cust_df["product_name"].mode()[0]
    }

    last10 = cust_df.sort_values("order_date").tail(10)

    stream_features = {
        "recent_order_count": len(last10),
        "recent_avg_amount": last10["total_amount"].mean(),
        "seconds_since_last_order": (pd.Timestamp.now()-last10["order_date"].max()).total_seconds(),
        "recent_top_category": last10["product_name"].mode()[0]
    }

    features[c] = {**batch_features, **stream_features}

for k,v in features.items():
    print(k,v)

In [ ]:
#Combining batch and stream features allows ML models to balance stable historical patterns with immediate behavioral shifts, which is essential for tasks like fraud detection where short-term anomalies override long-term norms.